# 01 — Data Cleaning: Dominick's Cereals

**目标（诊断优先，不急着建模）**

1. 用 memory-efficient dtype map 加载 `wcer.csv` 和 `upccer.csv`
2. 跑完 `rawData/README.md §3.8` 列出的全部 validation assertions
3. 对 `PRICE × MOVE / QTY = revenue` 手算一行 sanity check
4. 生成三个产出：
   - `data/processed/sku_store_week_panel.parquet` —— 清洗后的 SKU-store-week 面板（保留 UPC-store-week 粒度，**不在此 notebook 聚合到 brand × size**）
   - `reports/data_cleaning_summary.md` —— 数据质量摘要
   - `reports/figures/data_quality_*.png` —— 诊断图

**本 notebook 不做**：brand 抽取、size_oz 解析、price_per_oz、competitor price、建模。这些在 `02_eda.ipynb` 做。

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
RAW = PROJECT_ROOT / 'rawData'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'reports'
FIGURES = REPORTS / 'figures'

for p in (PROCESSED, FIGURES):
    p.mkdir(parents=True, exist_ok=True)

summary_lines: list[str] = []
def log(msg: str) -> None:
    print(msg)
    summary_lines.append(msg)

log(f'# Data Cleaning Summary — Dominick\'s Cereals')
log(f'Generated: {datetime.now().isoformat(timespec="seconds")}')
log('')

## 1. Load raw data

显式 dtype map + `usecols` 跳过 HEX 列，把 wcer 从 ~4 GB 压到 ~180 MB。

In [ ]:
WCER_DTYPES = {
    'STORE':  'int16',
    'UPC':    'int64',
    'WEEK':   'int16',
    'MOVE':   'int32',
    'QTY':    'int8',
    'PRICE':  'float32',
    'SALE':   'category',
    'PROFIT': 'float32',
    'OK':     'int8',
}
WCER_USECOLS = list(WCER_DTYPES.keys())  # exclude PRICE_HEX, PROFIT_HEX

wcer = pd.read_csv(
    RAW / 'wcer.csv',
    dtype=WCER_DTYPES,
    usecols=WCER_USECOLS,
    na_values=[''],
)
wcer['UPC'] = wcer['UPC'].astype('category')

log(f'## 1. Load')
log(f'- `wcer.csv`: {len(wcer):,} rows × {wcer.shape[1]} cols, '
    f'memory ~{wcer.memory_usage(deep=True).sum() / 1e6:.0f} MB')

In [ ]:
UPCCER_DTYPES = {
    'COM_CODE': 'int32',
    'UPC':      'int64',
    'DESCRIP':  'string',
    'SIZE':     'string',
    'CASE':     'int16',
    'NITEM':    'int64',
}
# upccer.csv contains legacy non-UTF-8 bytes (e.g. 0xd5 = 'Õ' in bundle marks like "PP .39Õ,1 OZ").
# Use latin-1 to decode without loss; we won't rely on these glyphs downstream.
upccer = pd.read_csv(RAW / 'upccer.csv', dtype=UPCCER_DTYPES, encoding='latin-1')
log(f'- `upccer.csv`: {len(upccer):,} rows × {upccer.shape[1]} cols')
upccer.head()

## 2. Schema validation

断言列集合、dtype、WEEK 范围。未来迁移到 `src/validation.py`。

In [ ]:
expected_wcer_cols = set(WCER_USECOLS)
assert set(wcer.columns) == expected_wcer_cols, (
    f'wcer columns mismatch: got {set(wcer.columns)}'
)
assert wcer['OK'].isin([0, 1]).all(), 'OK flag not in {0,1}'
assert wcer['WEEK'].between(1, 400).all(), 'WEEK out of [1,400]'
assert wcer['QTY'].ge(1).all(), 'QTY < 1 found'
assert wcer['MOVE'].ge(0).all(), 'negative MOVE found'

expected_upccer_cols = {'COM_CODE', 'UPC', 'DESCRIP', 'SIZE', 'CASE', 'NITEM'}
assert set(upccer.columns) == expected_upccer_cols
assert upccer['UPC'].is_unique, 'UPC not unique in upccer'

log('## 2. Schema validation')
log('- All schema assertions passed')

## 3. Row-level diagnostics — `wcer`

In [ ]:
log('## 3. Row-level diagnostics (wcer)')

ok_counts = wcer['OK'].value_counts().sort_index()
ok_pct = wcer['OK'].value_counts(normalize=True).sort_index() * 100
log(f'- OK flag: {ok_counts.to_dict()} ({ok_pct.round(2).to_dict()} %)')

log(f'- MOVE: zero-move rows = {(wcer["MOVE"] == 0).mean() * 100:.1f}%, '
    f'max = {wcer["MOVE"].max()}, p99 = {wcer["MOVE"].quantile(0.99):.0f}')

log(f'- QTY: bundle rows (QTY>1) = {(wcer["QTY"] > 1).mean() * 100:.2f}%, '
    f'distinct QTY values = {sorted(wcer["QTY"].unique().tolist())}')

log(f'- PRICE: zero-price rows = {(wcer["PRICE"] == 0).mean() * 100:.2f}%, '
    f'max = ${wcer["PRICE"].max():.2f}, p99 = ${wcer["PRICE"].quantile(0.99):.2f}')

profit_neg = (wcer['PROFIT'] < 0).mean() * 100
profit_hi = (wcer['PROFIT'] > 99).mean() * 100
profit_nan = wcer['PROFIT'].isna().mean() * 100
log(f'- PROFIT: median = {wcer["PROFIT"].median():.1f}%, '
    f'< 0 = {profit_neg:.2f}%, > 99 = {profit_hi:.2f}%, NaN = {profit_nan:.2f}%')

sale_counts = wcer['SALE'].value_counts(dropna=False)
log(f'- SALE value counts (incl. NaN): {sale_counts.to_dict()}')
sale_unknown = set(wcer['SALE'].dropna().unique()) - {'B', 'C', 'S'}
if sale_unknown:
    log(f'  **WARNING**: unexpected SALE codes: {sale_unknown}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

wcer.loc[wcer['PRICE'] > 0, 'PRICE'].clip(upper=15).hist(bins=60, ax=axes[0, 0])
axes[0, 0].set_title('PRICE distribution (clipped at $15)')
axes[0, 0].set_xlabel('PRICE ($, bundle)')

wcer.loc[(wcer['MOVE'] > 0) & (wcer['MOVE'] < 200), 'MOVE'].hist(bins=60, ax=axes[0, 1])
axes[0, 1].set_title('MOVE distribution (MOVE>0, clipped at 200)')
axes[0, 1].set_xlabel('units sold / week')

wcer.loc[wcer['PROFIT'].between(-5, 75), 'PROFIT'].hist(bins=60, ax=axes[1, 0])
axes[1, 0].set_title('PROFIT distribution (% gross margin)')
axes[1, 0].set_xlabel('PROFIT (%)')

sale_counts.head(10).plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('SALE code frequency')
axes[1, 1].set_xlabel('SALE')

fig.tight_layout()
fig.savefig(FIGURES / 'data_quality_wcer.png', dpi=120)
plt.show()

## 4. Uniqueness check — `(UPC, STORE, WEEK)`

In [ ]:
dup_mask = wcer.duplicated(subset=['UPC', 'STORE', 'WEEK'], keep=False)
n_dup = int(dup_mask.sum())
log('## 4. Uniqueness')
log(f'- duplicate (UPC, STORE, WEEK) rows: {n_dup:,}')
if n_dup:
    log(f'  **DECISION REQUIRED**: investigate before dedup')

## 5. UPC file diagnostics

**关键**：`COM_CODE` 可能有多个值。manual §3.2 原文："a single file can contain more than one commodity code"。

In [ ]:
log('## 5. UPC file diagnostics')
com_counts = upccer['COM_CODE'].value_counts()
log(f'- COM_CODE value_counts: {com_counts.to_dict()}')

size_match = upccer['SIZE'].str.extract(r'([\d.]+)\s*OZ', expand=False)
size_match_rate = size_match.notna().mean() * 100
log(f'- SIZE: OZ-parseable rate = {size_match_rate:.1f}%, '
    f'distinct non-OZ SIZE values = {sorted(upccer.loc[size_match.isna(), "SIZE"].dropna().unique().tolist())}')

desc_special = {c: upccer['DESCRIP'].str.contains(c, regex=False, na=False).sum()
                for c in ['#', '<', '~', '$', '*']}
log(f'- DESCRIP special char counts: {desc_special}')

## 6. Apply cleaning filters

过滤规则（可在 notebook 输出后调整）：
- `OK = 1`
- `PROFIT ∈ [0, 99]`（manual 定义范围之外视为缺失）
- `PRICE > 0`
- 保留 `MOVE = 0` 的行（零销量周对库存/stockout 故事有用；log 回归阶段再过滤）

In [ ]:
n0 = len(wcer)
wcer_clean = wcer[
    (wcer['OK'] == 1) &
    wcer['PROFIT'].between(0, 99) &
    (wcer['PRICE'] > 0)
].copy()
n1 = len(wcer_clean)
log('## 6. Cleaning filters applied')
log(f'- before: {n0:,} rows')
log(f'- after (OK=1 & 0≤PROFIT<99 & PRICE>0): {n1:,} rows '
    f'({n1 / n0 * 100:.1f}% retained, {n0 - n1:,} dropped)')

## 7. Derive fields + manual sanity check

Manual 公式：
- `unit_price = PRICE / QTY`
- `unit_cost = unit_price × (1 - PROFIT/100)`
- `revenue = PRICE × MOVE / QTY = unit_price × MOVE`

**手算验证**：挑一条 `QTY > 1` 且 `MOVE > 0` 的行，肉眼对照。

In [ ]:
wcer_clean['unit_price'] = wcer_clean['PRICE'].astype('float32') / wcer_clean['QTY']
wcer_clean['unit_cost']  = wcer_clean['unit_price'] * (1 - wcer_clean['PROFIT'] / 100)
wcer_clean['revenue']    = wcer_clean['unit_price'] * wcer_clean['MOVE']

# SALE cleaning: B/C/S documented; G observed but undocumented (→ unknown_promo);
# L only 1 row in raw — treated as data error (→ missing).
SALE_MAP = {
    'B': 'bonus_buy',
    'C': 'coupon',
    'S': 'price_reduction',
    'G': 'unknown_promo',
    'L': 'missing',
}
wcer_clean['sale_code_raw']   = wcer_clean['SALE'].astype('string')  # preserve original char
wcer_clean['sale_type_clean'] = (
    wcer_clean['SALE'].astype('string').map(SALE_MAP).fillna('missing').astype('category')
)
wcer_clean['promo'] = wcer_clean['sale_type_clean'] != 'missing'

sale_type_counts = wcer_clean['sale_type_clean'].value_counts().to_dict()
log(f'- sale_type_clean counts (post-filter): {sale_type_counts}')

bundle_rows = wcer_clean[(wcer_clean['QTY'] > 1) & (wcer_clean['MOVE'] > 0)]
if len(bundle_rows):
    row = bundle_rows.iloc[0]
    manual_revenue = row['PRICE'] * row['MOVE'] / row['QTY']
    log('## 7. Sanity check (bundle formula)')
    log(f'- sample row: UPC={int(row["UPC"])}, STORE={int(row["STORE"])}, '
        f'WEEK={int(row["WEEK"])}, PRICE={row["PRICE"]:.2f}, '
        f'QTY={int(row["QTY"])}, MOVE={int(row["MOVE"])}')
    log(f'- manual formula revenue = {manual_revenue:.2f}')
    log(f'- derived column revenue = {row["revenue"]:.2f}')
    assert np.isclose(manual_revenue, row['revenue'], rtol=1e-4), 'bundle formula mismatch'
    log('- **PASS**: derived revenue matches manual formula')
else:
    log('## 7. Sanity check: no bundle rows with MOVE>0 found (unexpected)')

# PROFIT sanity range (not a hard filter — hard filter is PROFIT ∈ [0, 99), applied in §6).
profit_median = float(wcer_clean['PROFIT'].median())
if 5 <= profit_median <= 30:
    log(f'- PROFIT median sanity: {profit_median:.1f}% ∈ [5, 30] → OK')
else:
    log(f'- **WARNING** PROFIT median = {profit_median:.1f}% outside [5, 30] sanity range — inspect before modeling')

## 8. Join wcer + upccer (metadata only, no aggregation)

保留 UPC-store-week 粒度。brand/size_oz 的解析留给 `02_eda.ipynb`。

In [ ]:
meta_cols = ['UPC', 'COM_CODE', 'DESCRIP', 'SIZE', 'NITEM']
wcer_clean['UPC_int'] = wcer_clean['UPC'].astype('int64')
panel = wcer_clean.merge(upccer[meta_cols], left_on='UPC_int', right_on='UPC',
                         how='left', suffixes=('', '_meta'))
panel = panel.drop(columns=['UPC_meta', 'UPC_int'])

n_missing = panel['DESCRIP'].isna().sum()
log('## 8. Join wcer × upccer')
log(f'- panel rows: {len(panel):,}')
log(f'- rows with no UPC metadata match: {n_missing:,} ({n_missing / len(panel) * 100:.2f}%)')

## 9. WEEK → calendar date mapping

Manual §5：Week 1 起始于 1989-09-14，每周 7 天。Week N 起始日 = `1989-09-14 + (N-1)*7`。

In [ ]:
WEEK_EPOCH = pd.Timestamp('1989-09-14')
week_to_date = pd.DataFrame({
    'WEEK': np.arange(1, 401, dtype='int16'),
    'week_start_date': [WEEK_EPOCH + pd.Timedelta(days=(w - 1) * 7) for w in range(1, 401)],
})

panel = panel.merge(week_to_date, on='WEEK', how='left')
panel['year'] = panel['week_start_date'].dt.year
panel['month'] = panel['week_start_date'].dt.month.astype('int8')

log('## 9. WEEK → date')
log(f'- date range: {panel["week_start_date"].min().date()} to '
    f'{panel["week_start_date"].max().date()}')
log(f'- years covered: {sorted(panel["year"].unique().tolist())}')

## 10. Granularity preview（不聚合，只看样本规模）

对比 UPC-store-week 与 brand 粒度聚合的样本损失。这里只 preview，**不改变 panel 粒度**。brand 抽取规则在 `02_eda.ipynb` 决定。

In [ ]:
n_upc_store_week = len(panel)
n_upc_store_pairs = panel.groupby(['UPC', 'STORE'], observed=True).ngroups
n_upc = panel['UPC'].nunique()
n_store = panel['STORE'].nunique()
n_week = panel['WEEK'].nunique()
weeks_per_pair = panel.groupby(['UPC', 'STORE'], observed=True).size()

log('## 10. Granularity preview')
log(f'- panel rows (UPC × store × week, after cleaning): {n_upc_store_week:,}')
log(f'- distinct UPC × store pairs: {n_upc_store_pairs:,}')
log(f'- distinct UPCs: {n_upc}')
log(f'- distinct stores: {n_store}')
log(f'- distinct weeks: {n_week}')
log(f'- weeks per UPC-store pair: median = {weeks_per_pair.median():.0f}, '
    f'p10 = {weeks_per_pair.quantile(0.1):.0f}, p90 = {weeks_per_pair.quantile(0.9):.0f}')

## 11. Save processed panel

In [ ]:
keep_cols = [
    'STORE', 'UPC', 'WEEK', 'week_start_date', 'year', 'month',
    'PRICE', 'QTY', 'MOVE', 'SALE', 'sale_code_raw', 'sale_type_clean', 'promo', 'PROFIT',
    'unit_price', 'unit_cost', 'revenue',
    'COM_CODE', 'DESCRIP', 'SIZE', 'NITEM',
]
panel_out = panel[keep_cols].copy()

out_path = PROCESSED / 'sku_store_week_panel.parquet'
panel_out.to_parquet(out_path, index=False, compression='snappy')
log('## 11. Saved processed panel')
log(f'- path: `{out_path.relative_to(PROJECT_ROOT)}`')
log(f'- size on disk: {out_path.stat().st_size / 1e6:.1f} MB')

## 12. Write cleaning summary

In [ ]:
summary_path = REPORTS / 'data_cleaning_summary.md'
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')
print(f'Summary written: {summary_path.relative_to(PROJECT_ROOT)}')
print(f'Figures written: {FIGURES.relative_to(PROJECT_ROOT)}/*.png')

---

## 下一步：`02_eda.ipynb`

- Brand 抽取规则（从 `DESCRIP` 前缀匹配 Kellogg's / General Mills / Post / Quaker / Ralston / Nabisco / PL）
- `size_oz` 正则解析，处理 `ASST` 与非 OZ 值
- `price_per_oz` 计算
- Granularity 决策：保留 UPC 还是聚合到 brand × size
- Competitor price 构造（同 store-week 其他品牌加权均价）
- Holiday week 标记（是否显式 dummy vs week FE 吸收）
- 可视化：价格-销量散点、时间序列、promo 分布等